# Part 1 - 실습 1: Amazon Bedrock 기초 & Foundation Model 탐색
**소요시간: 50분** | 난이도: ⭐⭐

## 학습 목표
1. Amazon Bedrock 클라이언트를 초기화하고 사용 가능한 Foundation Model 목록을 조회합니다.
2. `invoke_model` API로 Claude에게 첫 번째 프롬프트를 전송합니다.
3. `converse` API(통합 인터페이스)로 멀티턴 대화를 구현합니다.

## Amazon Bedrock 아키텍처
```
사용자 코드
   │
   ├─ bedrock          (모델 목록 조회)
   ├─ bedrock-runtime  (텍스트 생성, 스트리밍)
   └─ bedrock-agent-runtime  (RAG / Knowledge Base)
```

## 주요 모델 ID (2025 기준)
```
anthropic.claude-3-5-sonnet-20241022-v2:0  ← 최신 고성능
anthropic.claude-3-haiku-20240307-v1:0    ← 빠르고 경제적
amazon.titan-text-premier-v1:0            ← AWS 자체 모델
meta.llama3-8b-instruct-v1:0              ← 오픈소스 계열
```


---
## 🏢 기업 시나리오 — 생성형 AI 도입 TF

당신은 회사의 **생성형 AI 도입 TF 담당자**입니다.
"GPT 같은 걸 우리 업무에도 써보자"는 요청을 받았습니다. 모델을 직접 학습시키지 않고 **API 호출만으로** 시작합니다.

이번 실습에서는 다음을 수행합니다.
1. **모델 조사** → 어떤 Foundation Model이 있는지 목록 조회 (list_foundation_models)
2. **첫 호출** → Claude에게 프롬프트를 보내 응답 받기 (invoke_model)
3. **멀티턴 대화** → 챗봇의 토대가 되는 대화 구현 (converse)

> 💡 모델 선택은 비용·속도·품질의 trade-off입니다. 보통 저렴·빠른 모델(Haiku)로 PoC를 시작하고, 품질이 부족하면 상위 모델로 올립니다.


In [ ]:
# ✅ [제공 코드] 환경 초기화
import boto3, json
import pandas as pd

# Bedrock 클라이언트 (모델 목록 조회용)
bedrock = boto3.client('bedrock', region_name='us-east-1')

# Bedrock Runtime 클라이언트 (텍스트 생성용)
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

print('✅ Bedrock 클라이언트 생성 완료')
print(f'  bedrock           : {bedrock.meta.service_model.service_name}')
print(f'  bedrock-runtime   : {bedrock_runtime.meta.service_model.service_name}')


## ✏️ TODO 1: Foundation Model 목록 조회

Bedrock에서 사용 가능한 Foundation Model 목록을 조회하고 텍스트 생성 모델만 필터링하세요.


In [ ]:
# ✏️ TODO 1: list_foundation_models API로 사용 가능한 모델을 조회하세요
response = bedrock.list_foundation_models(
    byOutputModality=_____    # ← 'TEXT'
)

models = response[_____]   # ← 'modelSummaries'
print(f'텍스트 생성 모델 수: {len(models)}개\n')

rows = []
for m in models:
    rows.append({
        '모델ID'    : m[_____],                 # ← 'modelId'
        '제공사'    : m[_____],                 # ← 'providerName'
        '모델명'    : m['modelName'],
    })

df = pd.DataFrame(rows)
print(df[df['모델ID'].str.contains('claude|titan|llama')].to_string(index=False))


## ✏️ TODO 2: invoke_model — 첫 번째 텍스트 생성

Claude에게 프롬프트를 전송하고 응답을 파싱하세요. `invoke_model`은 모델별 고유 페이로드 구조를 사용합니다.

```python
# Claude 3 페이로드 구조
{
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 1024,
    'messages': [{'role': 'user', 'content': '프롬프트 내용'}]
}
```


In [ ]:
# ✏️ TODO 2: invoke_model API로 Claude에게 질문을 전송하세요
# us-east-1이면 프리픽스는 us. 입니다.
MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

payload = {
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': _____,           # ← 1024
    'messages': [
        {
            'role': _____,         # ← 'user'
            'content': _____       # ← 'AWS Bedrock을 한 문장으로 설명해줘.'
        }
    ]
}

response = bedrock_runtime.invoke_model(
    modelId=_____,                 # ← MODEL_ID
    body=json.dumps(_____),        # ← payload
    contentType='application/json'
)

result = json.loads(response['body'].read())
answer = result[_____][0][_____]   # ← 'content', 'text'
print('Claude 응답:')
print(answer)


In [ ]:
# ✏️ TODO 2-2: invoke_model API로 Nova에게 질문을 전송하세요
MODEL_ID = 'amazon.nova-micro-v1:0'

payload = {
    'schemaVersion': 'messages-v1',    # Nova 전용 필드
    'inferenceConfig': {'maxTokens': 1024},  # 힌트: 응답으로 생성할 최대 토큰 수 (정수)
    'messages': [
        {
            'role': 'user',             # 힌트: 대화에서 사람 쪽 역할을 나타내는 문자열
            'content': [{'text': 'AWS Bedrock을 한 문장으로 설명해줘.'}]  # 힌트: 모델에게 보낼 프롬프트 문자열을 입력하세요
        }
    ]
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,                     # 힌트: 이 셀 상단에 정의된 모델 ID 변수를 참조하세요
    body=json.dumps(payload),             # 힌트: 위에서 정의한 페이로드 딕셔너리를 JSON 직렬화해 전달하세요
    contentType='application/json'
)

result = json.loads(response['body'].read())
answer = result['output']['message']['content'][0]['text']
print('Nova 응답:')
print(answer)

In [ ]:
# ✅ [제공 코드] Temperature & Max Tokens 파라미터 실험
# ⚠️ 이 셀은 그대로 실행하면 ValidationException이 발생합니다. 왜 그런지 생각해 보세요!
#    힌트: 직전 셀(TODO 2-2)에서 MODEL_ID가 'amazon.nova-micro-v1:0'으로 재할당되었는데,
#          아래 payload는 Claude 전용 형식(anthropic_version, temperature, ...)입니다.
#          Nova는 이 키들을 모르기 때문에 "extraneous key is not permitted" 에러를 냅니다.
#    해결: 아래 줄의 주석을 해제해 MODEL_ID를 Claude로 되돌린 뒤 다시 실행하세요.
# MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

def call_claude(prompt, temperature=0.7, max_tokens=512):
    payload = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': max_tokens,
        'temperature': temperature,
        'messages': [{'role': 'user', 'content': prompt}]
    }
    resp = bedrock_runtime.invoke_model(
        modelId=MODEL_ID, body=json.dumps(payload), contentType='application/json'
    )
    return json.loads(resp['body'].read())['content'][0]['text']

prompt = '파이썬의 장점을 세 가지만 알려줘.'
for temp in [0.1, 0.7, 1.2]:
    print(f'\n--- temperature={temp} ---')
    print(call_claude(prompt, temperature=temp))


## ✏️ TODO 3: converse API — 통합 멀티턴 대화

`converse`는 모델별 페이로드 차이를 추상화한 통합 API입니다. 대화 히스토리를 누적하여 멀티턴을 구현하세요.

```python
# converse 구조
bedrock_runtime.converse(
    modelId='...',
    messages=[{'role': 'user', 'content': [{'text': '...'}]}],
    inferenceConfig={'maxTokens': 512, 'temperature': 0.7}
)
```


In [ ]:
# ✏️ TODO 3: converse API로 멀티턴 대화를 구현하세요
# 💡 converse API는 모델별 payload 형식이 필요 없는 통합 인터페이스입니다.
#    invoke_model과 달리 Claude든 Nova든 아래 코드는 그대로 두고 MODEL_ID만 바꾸면 동작합니다.
#    두 모델로 각각 실행해 보세요.
MODEL_ID = 'amazon.nova-micro-v1:0'
# MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

conversation_history = []

def chat(user_message, system_prompt=None):
    """대화 히스토리를 유지하며 converse API를 호출합니다"""
    conversation_history.append({
        'role': _____,                           # ← 'user'
        'content': [{'text': _____}]             # ← user_message
    })

    kwargs = {
        'modelId': MODEL_ID,
        'messages': _____,                       # ← conversation_history
        'inferenceConfig': {'maxTokens': 512, 'temperature': 0.7}
    }
    if system_prompt:
        kwargs['system'] = [{'text': system_prompt}]

    response = bedrock_runtime.converse(**kwargs)

    assistant_msg = response['output'][_____]['content'][0][_____]  # ← 'message', 'text'
    conversation_history.append({
        'role': 'assistant',
        'content': [{'text': assistant_msg}]
    })
    return assistant_msg

# 멀티턴 대화 테스트
SYSTEM = '당신은 친절한 AI 어시스턴트입니다. 한국어로 답변하세요.'

print('사용자:', '안녕하세요! 클라우드 컴퓨팅이 뭔가요?')
print('AI:', chat('안녕하세요! 클라우드 컴퓨팅이 뭔가요?', SYSTEM))
print()
print('사용자:', 'AWS와 Azure 중 어떤 게 더 좋아요?')
print('AI:', chat('AWS와 Azure 중 어떤 게 더 좋아요?'))
print()
print(f'대화 히스토리 길이: {len(conversation_history)}개 메시지')


---
## 🔗 실무로 연결하기

`사용자 입력` → `Lambda` → `Bedrock(converse)` → `응답` 의 흐름이 모든 생성형 AI 앱의 기본 골격입니다.

- **모델 선택 기준**: Haiku(빠름·저렴, 대량 처리) / Sonnet·Opus(고품질, 복잡 추론). 비용은 토큰 사용량 기반.
- PoC는 저렴한 모델로 시작 → 품질 검증 후 상위 모델 또는 프롬프트 개선.
- `converse` API는 모델이 바뀌어도 코드를 거의 그대로 재사용할 수 있어 실무에서 선호됩니다.


## 💡 심화 도전
1. `meta.llama3-8b-instruct-v1:0`을 `converse`로 호출하여 Claude와 같은 질문에 대한 응답을 비교해보세요.
2. Temperature를 0.01로 설정했을 때와 1.5로 설정했을 때 동일 프롬프트 결과가 어떻게 달라지는지 확인하세요.
3. `converse`의 `system` 파라미터로 다른 성격의 AI 페르소나를 설정해보세요.


## ✅ 정답 코드

👆 모두 풀고 난 후 확인하세요

```python
# TODO 1
response = bedrock.list_foundation_models(byOutputModality='TEXT')
models = response['modelSummaries']
m['modelId'], m['providerName']

# TODO 2
payload = {
    'anthropic_version': 'bedrock-2023-05-31',
    'max_tokens': 1024,
    'messages': [{'role': 'user', 'content': 'AWS Bedrock을 한 문장으로 설명해줘.'}]
}
response = bedrock_runtime.invoke_model(modelId=MODEL_ID, body=json.dumps(payload), ...)
answer = result['content'][0]['text']

# TODO 3
conversation_history.append({'role': 'user', 'content': [{'text': user_message}]})
kwargs = {'modelId': MODEL_ID, 'messages': conversation_history, ...}
assistant_msg = response['output']['message']['content'][0]['text']
```
